# Chargement du modèle et du tokenizer

In [1]:
from model import Model

model = Model(max_content = 3)

h:\Documents\GitHub\NLP_narrative-game\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 128.02it/s]


In [2]:
system_content = "Tu es le Game Master d'un jeu de rôle fantasy en solo. Tu décris l'univers, les conséquences des actions du joueur et proposes toujours une liste d'actions possibles sous la forme A, B ou C. Le joueur doit obligatoirement choisir l'une des actions proposées. Si le joueur propose une action qui n'est pas dans la liste, indique clairement qu'elle est impossible et rappelle les actions disponibles. Les descriptions doivent être courtes, de 1 ou 2 phrases. "

# Paramètres

Le format de conversation attendu est le chat template mentionné dans le notebook d'observation du modèle: 
un content pour le role system, puis des succèssion de contents pour les roles "user" et "assistant". 

max_new_tokens = 512: cette longueur me semble nécessaire pour un agent jouant le rôle du game master, ni trop longue demandant trop de temps d'exécution <br>
do_sample = True: nous utilisons l'agent notamment pour sa créativité <br>
temperature = 1: nous utilisons la température par défaut, le hasard est un élément important, mais pas besoin de l'augmenter non plus pour que le modèle reste contrôlé <br>
top-k = 30: nous gardons les 30 tokens les plus probables pour un filtre correcte avec une certaine incertitude pour avoir des scénariots différents <br>
top-p = 0.8: nous filtrons les tokens les moins probables. 

# Test

## Zero-shot

### Action valide

In [ ]:
model.init_history(system_content)

In [26]:
model.add_content("assistant", "Vous arrivez devant une vieille porte en bois. Trois symboles sont gravés dessus : une lune, un soleil et une étoile. Vous pouvez : A) toucher le symbole de la lune B) toucher le symbole du soleil C) toucher le symbole de l'étoile.")

model.add_content("user", "Je choisi A")

model.generate()

print(model.assistant_content)

La porte s'ouvre lentement sur un vieil auberge dans le désert. Dans l'intérieur, vous apercevez un vieux baron tenant son marteau à deux manchettes. Il se redresse doucement de sa chaise et vous observe sans sourire. Il murmure : "Bienvenue à mon domaine... si je peux l'appeler ainsi." Il saisit votre épée et vous ordonne de rentrer chez-vous. Que feriez-vous ensuite ? A) Rencontrer le Baron pour discuter de votre arrivée B) Faire demi-tour et repartir immédiatement C) Dormir quelques jours dans votre nouveau quartier.


In [27]:
model.add_content("user", "Je choisi A")

model.generate()

print(model.assistant_content)

Le Baron s'incline et fait signe que vous pouvez entrer. Une pièce éclairée par la lumière orangée d'une lampe de chevet vous accueille. Sur une table basse, un verre d'eau glacée et une assiette pleine de biscuits dorés sont disposés. Vous sentez le confort, mais aussi l'hostilité sous ces gestes civils. Quel choix faisiez-vous ? A) Manger quelque chose avant d'enquêter B) Demander aux serviteurs de vous apporter un réconfort C) Rester là pour voir comment cela va se passer.


Le model a la réaction attendu, avec une liste des 3 choix ainsi qu'une succession logique des événements. 

### Action invalide

In [28]:
model.init_history(system_content)

In [29]:
model.add_content("assistant", "Vous arrivez devant une vieille porte en bois. Trois symboles sont gravés dessus : une lune, un soleil et une étoile. Vous pouvez : A) toucher le symbole de la lune B) toucher le symbole du soleil C) toucher le symbole de l'étoile.")

model.add_content("user", "Je fait demi tour. ")

model.generate()

print(model.assistant_content)

Bien joué! Le symbole de la lune semble le plus proche maintenant.


Le modèle ne contredit pas l'utilisateur quand il fait une action invalide. 

## Few-shot

## Action valide

In [19]:
exemples = """
Les exemples suivants sont indépendants les uns des autres.
Ils illustrent le comportement attendu du Game Master.
Ils ne font pas partie de la partie actuelle.

================ EXEMPLE 1 : ACTION NORMALE ================

Assistant :
Vous êtes dans un bureau sombre. Une porte se trouve au nord,
un bureau est placé contre le mur et une fenêtre donne sur la rue.

A. Examiner la porte
B. Fouiller le bureau
C. Regarder par la fenêtre

Utilisateur :
B

Assistant :
Vous fouillez le bureau et découvrez une petite lampe de poche.
Vous la prenez et la rangez dans votre inventaire.

A. Allumer la lampe
B. Examiner la porte
C. Regarder par la fenêtre


================ EXEMPLE 2 : ACTION INCOHÉRENTE ================

Assistant :
Vous êtes dans un couloir d'un immeuble moderne.
Une porte verrouillée se trouve devant vous.

A. Examiner la serrure
B. Chercher une autre sortie
C. Retourner dans la pièce précédente

Utilisateur :
Je lance un sort magique pour téléporter la porte ailleurs.

Assistant :
La magie n'existe pas dans cet univers moderne. Vous ne pouvez
donc pas téléporter la porte.

A. Examiner la serrure
B. Chercher un objet utile
C. Explorer la pièce précédente
"""

In [22]:
model.init_history(system_content)

model.add_content("user", exemples)

In [23]:
model.add_content("assistant", "Vous arrivez devant une vieille porte en bois. Trois symboles sont gravés dessus : une lune, un soleil et une étoile. Vous pouvez : A) toucher le symbole de la lune B) toucher le symbole du soleil C) toucher le symbole de l'étoile.")

model.add_content("user", "Je choisi A")

model.generate()

print(model.assistant_content)

Le symbole de la lune a brûlé violemment au contact de vos doigts. Un nuage de poussière blanche s’est levé autour de vous. Quelle action prendrez-vous ? A) rentrer chez vous B) retourner à votre point de départ C) continuer votre voyage.


Le model a la réaction attendu, avec une liste des 3 choix ainsi qu'une succession logique des événements. 

Action invalide

In [6]:
model.init_history(system_content)

model.add_content("user", exemples)

In [7]:
model.add_content("assistant", "Vous arrivez devant une vieille porte en bois. Trois symboles sont gravés dessus : une lune, un soleil et une étoile. Vous pouvez : A) toucher le symbole de la lune B) toucher le symbole du soleil C) toucher le symbole de l'étoile.")

model.add_content("user", "Je fait demi tour. ")

model.generate()

print(model.assistant_content)

Votre choix semble avoir eu un effet mystérieux. De nouveaux éléments suggèrent de se pencher pour découvrir quelque chose derrière cette porte... Avez-vous décidé quoi faire maintenant ?


Le modèle ne contredit pas l'utilisateur quand il fait une action invalide. 